# Quasi-Newton methods

`QuasiNewton` never forms a Hessian. It keeps an approximation of the *inverse*
Hessian and refreshes it each iteration with a rank-2 update — `dfp` or `bfgs`
— chosen through the `update=` argument.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from mopt.nonlinear import NLPProblem, QuasiNewton, bfgs, dfp

## Test problem

The same shape of quartic as elsewhere in these examples. Note that no `hess`
is supplied to `NLPProblem` — quasi-Newton methods do not need one.

In [2]:
Q = np.array([
    [1.99, 0.04, -0.02, 0.17, 0.04],
    [0.04, 1.88, -0.08, -0.03, 0.01],
    [-0.02, -0.08, 1.95, -0.15, 0.01],
    [0.17, -0.03, -0.15, 1.33, 0.14],
    [0.04, 0.01, 0.01, 0.14, 1.78],
])
M = np.array([
    [1.95, 0.97, -0.41, -0.35, -0.46],
    [0.97, 2.69, -0.83, -0.25, -0.16],
    [-0.41, -0.83, 1.99, -0.63, -0.22],
    [-0.35, -0.25, -0.63, 3.15, 0.02],
    [-0.46, -0.16, -0.22, 0.02, 3.33],
])
b = np.array([-0.7, 2.0, 1.2, -1.1, -1.9])
c = np.array([0.3, 1.0, -1.7, 1.2, 1.1])

f = lambda x: float((x @ Q @ x + b @ x) ** 2 + x @ M @ x + c @ x)
grad_f = lambda x: (4.0 * (Q @ x) + 2.0 * b) * (x @ Q @ x + b @ x) + 2.0 * (M @ x) + c

## Both updates, from several starts and several initial approximations

`H0` seeds the inverse-Hessian approximation. The default is the identity,
which makes the first step plain steepest descent; any positive definite matrix
is allowed.

In [3]:
def random_pd(rng, n=5):
    L = np.tril(rng.uniform(-1.0, 1.0, size=(n, n)))
    np.fill_diagonal(L, rng.uniform(0.5, 1.5, size=n))
    return L @ L.T

rng = np.random.default_rng(42)
inits = np.vstack([np.zeros(5), rng.uniform(-2.0, 2.0, size=(5, 5))])
H_inits = [("I", None)] + [(f"chol_{j}", random_pd(rng)) for j in range(3)]

rows = []
for x0 in inits:
    problem = NLPProblem(f=f, x0=x0, grad=grad_f)
    x_ref = minimize(f, x0, jac=grad_f, method="BFGS").x
    for h_name, H0 in H_inits:
        for update in (dfp, bfgs):
            result = QuasiNewton(update=update, H0=H0, max_iter=5000).solve(problem)
            assert result.success, result.message
            np.testing.assert_allclose(result.x, x_ref, atol=1e-5)
            rows.append({
                "x0": tuple(np.round(x0, 3)),
                "H_0": h_name,
                "method": update.__name__,
                "iters": result.n_iter,
                "f": result.fun,
                "|x - x_scipy|": np.linalg.norm(result.x - x_ref),
            })

table = (pd.DataFrame(rows)
         .pivot(index=["x0", "H_0"], columns="method")
         .swaplevel(axis=1)
         .reindex(columns=pd.MultiIndex.from_product(
             [["dfp", "bfgs"], ["iters", "f", "|x - x_scipy|"]])))
table

dfp                          \
                                             iters         f |x - x_scipy|   
x0                                    H_0                                    
(-1.091, 0.218, -1.745, 1.311, 0.527) I         59 -0.376705  2.202550e-06   
                                      chol_0    35 -0.376705  2.137621e-06   
                                      chol_1    36 -0.376705  2.142291e-06   
                                      chol_2    30 -0.376705  2.190680e-06   
(-0.517, 1.707, 0.575, 1.291, -0.226) I         55 -0.376705  7.495788e-07   
                                      chol_0    41 -0.376705  6.557195e-07   
                                      chol_1    41 -0.376705  6.931498e-07   
                                      chol_2    43 -0.376705  6.831092e-07   
(0.0, 0.0, 0.0, 0.0, 0.0)             I         22 -0.376705  2.013726e-06   
                                      chol_0    13 -0.376705  2.001032e-06   
                                      chol_1    12 -0.376705  2.059762e-06   
                                      chol_2    11 -0.376705  2.062091e-06   
(1.032, -0.582, 1.883, 1.572, 1.114)  I         53 -0.376705  6.655131e-07   
                                      chol_0    29 -0.376705  6.824306e-07   
                                      chol_1    43 -0.376705  6.048511e-07   
                                      chol_2    53 -0.376705  6.394866e-07   
(1.096, -0.244, 1.434, 0.789, -1.623) I         75 -0.376705  8.816041e-08   
                                      chol_0    40 -0.376705  2.251908e-07   
                                      chol_1    45 -0.376705  2.643197e-07   
                                      chol_2    34 -0.376705  1.564587e-07   
(1.902, 1.045, 1.144, -1.488, -0.198) I        103 -0.376705  1.307736e-06   
                                      chol_0    74 -0.376705  1.285925e-06   
                                      chol_1    35 -0.376705  1.271565e-06   
                                      chol_2    74 -0.376705  1.378045e-06   

                                              bfgs                          
                                             iters         f |x - x_scipy|  
x0                                    H_0                                   
(-1.091, 0.218, -1.745, 1.311, 0.527) I         25 -0.376705  2.145472e-06  
                                      chol_0    23 -0.376705  2.101486e-06  
                                      chol_1    23 -0.376705  2.139749e-06  
                                      chol_2    23 -0.376705  2.067873e-06  
(-0.517, 1.707, 0.575, 1.291, -0.226) I         29 -0.376705  6.986424e-07  
                                      chol_0    28 -0.376705  6.965316e-07  
                                      chol_1    25 -0.376705  7.272687e-07  
                                      chol_2    25 -0.376705  7.051249e-07  
(0.0, 0.0, 0.0, 0.0, 0.0)             I         14 -0.376705  2.009505e-06  
                                      chol_0    11 -0.376705  2.016379e-06  
                                      chol_1    12 -0.376705  2.030347e-06  
                                      chol_2    12 -0.376705  2.005281e-06  
(1.032, -0.582, 1.883, 1.572, 1.114)  I         27 -0.376705  6.408031e-07  
                                      chol_0    24 -0.376705  6.230933e-07  
                                      chol_1    25 -0.376705  6.359229e-07  
                                      chol_2    27 -0.376705  6.272165e-07  
(1.096, -0.244, 1.434, 0.789, -1.623) I         30 -0.376705  1.908436e-07  
                                      chol_0    25 -0.376705  2.046457e-07  
                                      chol_1    24 -0.376705  2.142748e-07  
                                      chol_2    22 -0.376705  1.967448e-07  
(1.902, 1.045, 1.144, -1.488, -0.198) I         28 -0.376705  1.305524e-06  
                                      chol_0    31 -0.376705  1.260752e-06  
               

In [4]:
print("mean iterations by update and H_0\n")
print(pd.DataFrame(rows).groupby(["method", "H_0"])["iters"].mean().unstack().round(1))

mean iterations by update and H_0

H_0        I  chol_0  chol_1  chol_2
method                              
bfgs    25.5    23.7    21.8    23.3
dfp     61.2    38.7    35.3    40.8


## No gradient either, if you do not have one

`NLPProblem.gradient` falls back to central finite differences, so the solver
runs on `f` alone — at the cost of `2n` extra function evaluations per
gradient.

In [5]:
result = QuasiNewton().solve(NLPProblem(f=f, x0=np.zeros(5)))   # no grad=
print(f"success={result.success}  iters={result.n_iter}  f={result.fun:.9f}")
print("matches the analytic-gradient run:",
      np.allclose(result.fun, min(r["f"] for r in rows if r["x0"] == (0.0, 0.0, 0.0, 0.0, 0.0)),
                  atol=1e-6))

success=True  iters=14  f=-0.376705107
matches the analytic-gradient run: True
